# Remove fillers: written ones (hm, a-a, ee) and unwritten sounds (ee/mm, laughs, coughs, stray words)
Input: `dataset_clean/` (after `03_denoise` and `04_remove_non_speech`), which is not changed.
Output: `dataset_nofillers/`, the same layout (`metadata.json` + `chunks/`); then step 5 cuts it into sentence chunks in `dataset_final/`, **the dataset for training**.
Both, and the work folder `outputs/fillers/`, are in the run folder that `RUN` in `pipeline_paths.py` sets (the project root, or `runs/<RUN>/`).

**Where to run:** on the DGX Spark, kernel `.venv-denoise`, where all the models are already cached. It also runs on Kaggle, where there is no `pipeline_paths.py`: give `DATASET`, `OUT` and `WORK` as absolute paths, see the config cell.
- Alignment: `facebook/mms-1b-all` with the Uzbek (Latin) adapter.
- Sound classes: the AST AudioSet classifier from 03.
- Vocal bursts (laughs, coughs, sneezes, sighs, gasps, cries): [Vocal Burst Locator v2](https://huggingface.co/laion/vocalburst-locator) (`model_v2.pt`, 972 MB). It gives a burst probability for every 20 ms. It says *where* a burst is; AST names it (laugh / cough / breath) when it can.

To keep it running after the terminal closes, run it headless in `tmux`:
`.venv-denoise/bin/python scripts/run_notebook.py pipeline/05_remove_fillers.ipynb`.

**1. Analyse** (`outputs/fillers/analysis.json`). Takes about 1.5–2 h for all chunks on the shared GPU, and can be resumed. For a quick look first, set `SCOPE = "written"`: it only analyses the ~600 chunks with fillers in their text.
- **Forced alignment:** MMS gives the probability of every letter every 20 ms, and the transcript is aligned to it, so every word gets a start and an end.
- **Written fillers:** for each filler word in the text (hm, mhm, a-a, e/ee/eee, mmm, um, uh, aha), its piece of audio is the stretch between the neighbouring words that has sound in it. CTC only marks a short spike for such a word, so its own span is too short.
- **Unwritten sounds:** stretches of sound, or of a vocal burst even when it's quiet, that no word covers. Words get `WORD_PAD` added on each side, because CTC marks them early and short. Each stretch gets measured:
  - the Vocal Burst Locator's probability;
  - what MMS hears in it by itself (greedy decoding), its voicing and pitch steadiness (pYIN);
  - AST's probabilities for laughter, cough and breathing.

**2. Decide** (fast, so you can change the settings and run it again). Each unwritten sound gets a type:
- `burst`: the Vocal Burst Locator hears a vocal burst (mean probability ≥ `BURST_MIN`). If AST also names it, it becomes `laugh`, `cough` or `breath` instead;
- `laugh`, `cough`, `breath`: AST hears it (≥ `AST_MIN`);
- `speech`: MMS, decoding the sound by itself, hears at least `MIN_HEARD_LETTERS` letters that are not a filler. These are words the transcript doesn't have, or the other speaker's "ha";
- `filler`: MMS hears "eee"/"mmm"/"hm", or it hears no letters and the sound is one steady voiced tone (pitch within `MAX_PITCH_ST` semitones): ee, mm, a-a, a drawn-out vowel;
- `breath`: mostly unvoiced;
- `word_edge`: a short voiced bit touching a word, most likely just that word's end. It's never cut;
- `other`: any other voiced sound without words.

Each chunk then gets one of three outcomes:
- **Dropped:**
  - text that is only fillers (`filler_only`);
  - text that can't be aligned (`align_failed`);
  - with `MAX_BURST_IN_WORDS_SEC` set, chunks with that much laughing or coughing on top of the words (`burst_over_words`). The cut can't remove those without cutting words;
  - `MIN_ALIGN_PROB`, if set;
  - at least `MIN_UNTRANSCRIBED_SEC` of `speech` (`untranscribed_speech`);
  - shorter than `MIN_SEC` after cutting.
- **Edited:**
  - the written fillers are cut out of the audio *and* removed from the text;
  - the unwritten sounds whose type is in `CUT_TYPES` are cut out of the audio (the text is unchanged).

  A cut inside a chunk is replaced by a short pause of the chunk's own room tone, so the words don't run together.
- **Kept** as it is.

**3. Review** (`outputs/fillers/review/`; open `index.html`): `N_REVIEW` examples of each kind, as before/after wavs plus the detected sound itself. Every dropped chunk is also linked into `outputs/fillers/dropped/<reason>/` (a hard link: no extra space).

**4. Build** `dataset_nofillers/`:
- Unchanged chunks are hard links, so they take no space; edited chunks are new wavs.
- `metadata.json` has the new `text` and `duration`, and a `fillers` field with the cuts and the old text.
- It's rebuilt from scratch on every run.

**5. Sentence chunks** `dataset_final/`, the dataset for training. 02 cut every speaker turn longer than 29.5 s at a pause, often in the middle of a sentence (about 30% of the chunks end mid-sentence). This step puts those pieces back together and cuts again only between sentences:
- **Joined:** chunks of the same speaker that follow each other directly in the audio (one starts where the other ends) and were denoised the same way.
- **Cut:** at every sentence end (`.` `!` `?` `…`) followed by a pause of at least `SPLIT_PAUSE_SEC`. Never inside a sentence.
- **Long:** a piece longer than `MAX_CHUNK_SEC` is cut at its sentence end with the longest pause (at least `MIN_CUT_PAUSE_SEC`). Long speech without such a sentence end keeps the cuts it had before 05.
- **Short:** a piece shorter than `MIN_CHUNK_SEC` joins its neighbour across the shorter pause; if that would make it too long, it's left out.
- **Edges:** every chunk keeps at most `EDGE_SILENCE_SEC` of silence before its first word and after its last word.
- The pauses come from a forced alignment (MMS, as in step 1) of the final audio and text, saved in `outputs/fillers/words.json`: after changing the settings only the cutting runs again (a minute). A pause is the time between two aligned words, `WORD_PAD` taken off.
- `metadata.json`: per chunk `file, speaker, text, duration`, `start`/`end` (the span of its sources in the original audio), `denoised`, and `sources`: the `dataset_nofillers/` chunks it was made from, with the seconds taken from each. Review: `outputs/fillers/sentences/index.html`.

In [1]:
import html
import importlib
import json
import os
import re
import shutil
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
# the project root (where pyproject.toml is); elsewhere (e.g. Kaggle) the current folder: give absolute paths below
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())

In [2]:
# The folders of the run in pipeline_paths.py. On Kaggle (no pipeline_paths.py) delete the P lines and give
# the folders as strings: DATASET = "/kaggle/input/<your-dataset>/dataset_clean",
# OUT = "/kaggle/working/dataset_nofillers", WORK = "/kaggle/working/fillers", FINAL = "/kaggle/working/dataset_final"
import pipeline_paths as P
importlib.reload(P)           # a change of RUN in pipeline_paths.py counts without restarting the kernel
print(P.describe())
DATASET = P.DATASET_CLEAN     # input folder (metadata.json + chunks/), not changed
OUT = P.DATASET_NOFILLERS     # output folder, rebuilt on every run of step 4
WORK = P.OUTPUTS / "fillers"  # analysis.json, report.json, review/, dropped/, words.json, sentences/
FINAL = P.DATASET_FINAL       # step 5: the sentence chunks, the dataset for training
SCOPE = "all"                 # "all" chunks, or "written": only those with fillers in the text (quick)
LIMIT = None                  # e.g. 200: only the first N chunks of the scope (a test)

# step 1 (changing these needs a new analysis: delete analysis.json)
SOUND_DB = 30.0               # a 20 ms frame is "sound" when at most this far below the chunk's speech level
WORD_PAD = (0.05, 0.12)       # seconds added before / after every aligned word (CTC marks words early and short)
MIN_GAP_SEC = 0.2             # shorter unwritten sounds are ignored
BURST_THRESHOLD = 0.5         # Vocal Burst Locator v2: a 20 ms frame is part of a burst at this probability (its card: 0.5)

# step 2
AST_MIN = 0.2                 # AST probability for laugh / cough / breath
MAX_PITCH_ST = 2.5            # a filler (ee, mm) keeps its pitch within this many semitones
MIN_HEARD_LETTERS = 3         # MMS hears this many letters in a sound by itself: words ("speech"), or "eee"/"mmm"
BURST_MIN = 0.5               # a sound is a vocal burst when the locator's mean probability over it is at least this
MAX_BURST_IN_WORDS_SEC = None # e.g. 0.5: drop chunks with this much burst (laughing / coughing) on top of words
CUT_TYPES = {"filler", "burst", "laugh", "cough", "speech", "other"}   # unwritten sounds cut out; "breath" stays
MIN_UNTRANSCRIBED_SEC = 0.5   # this much "speech" that the text does not have: drop the chunk
MIN_ALIGN_PROB = None         # e.g. 0.5: drop chunks whose text aligns badly (mean letter probability)
MIN_SEC = 1.0                 # drop a chunk shorter than this after cutting
PAUSE_SEC = (0.10, 0.35)      # a cut inside a chunk becomes 40% of its length in room tone, within these limits
FADE_SEC = 0.01

# step 5: sentence chunks
SPLIT_PAUSE_SEC = 0.5         # a sentence end followed by at least this much pause is always a cut
MAX_CHUNK_SEC = 30.0          # a longer piece is cut at its sentence end with the longest pause...
MIN_CUT_PAUSE_SEC = 0.15      # ...if that pause is at least this long; else it keeps its old cuts
MIN_CHUNK_SEC = 1.0           # a shorter piece joins its neighbour, or is left out
EDGE_SILENCE_SEC = 0.2        # silence kept before the first word and after the last word of a chunk

N_REVIEW = 8
SAVE_EVERY = 200

run 'test': reading and writing in /home/intern/projects/tts/runs/test


## Models and functions

In [3]:
dev = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
ds, out_dir, work = root / DATASET, root / OUT, root / WORK
meta = json.load(open(ds / "metadata.json", encoding="utf-8"))
SR = meta["sample_rate"]
chunks = meta["chunks"]
by_file = {c["file"]: c for c in chunks}

from transformers import ASTFeatureExtractor, ASTForAudioClassification, AutoProcessor, Wav2Vec2ForCTC

MMS, LANG = "facebook/mms-1b-all", "uzb-script_latin"
processor = AutoProcessor.from_pretrained(MMS, target_lang=LANG)
mms = Wav2Vec2ForCTC.from_pretrained(MMS, target_lang=LANG, ignore_mismatched_sizes=True).eval().to(dev)
VOCAB = processor.tokenizer.get_vocab()
BLANK = processor.tokenizer.pad_token_id
LETTERS = set("abcdefghijklmnopqrstuvwxyz'") & set(VOCAB)
FRAME = 0.02                  # MMS: one frame per 20 ms (320 samples at 16 kHz)
HOP = 320

AST_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_fe = ASTFeatureExtractor.from_pretrained(AST_NAME)
ast = ASTForAudioClassification.from_pretrained(AST_NAME).eval().to(dev)
GROUPS = {"laugh": ["Laughter", "Baby laughter", "Giggle", "Snicker", "Belly laugh", "Chuckle, chortle"],
          "cough": ["Cough", "Throat clearing", "Sneeze", "Sniff"],
          "breath": ["Breathing", "Sigh", "Gasp", "Pant", "Wheeze", "Snoring"]}
GROUP_IDS = {g: [ast.config.label2id[n] for n in names if n in ast.config.label2id] for g, names in GROUPS.items()}

# Vocal Burst Locator v2 (laion/vocalburst-locator, Apache-2.0): Whisper-small encoder + a small head, one burst
# probability per 20 ms over a 30 s window (laughs, coughs, sneezes, sighs, gasps, cries). The class is the one in the
# repo's inference.py; the Whisper part is built from its config only, since model_v2.pt holds all the weights.
from huggingface_hub import hf_hub_download
from transformers import WhisperConfig, WhisperFeatureExtractor, WhisperModel


class WhisperSegmenter(torch.nn.Module):
    def __init__(self, whisper_id="openai/whisper-small"):
        super().__init__()
        self.whisper = WhisperModel(WhisperConfig.from_pretrained(whisper_id))
        d_model = self.whisper.config.d_model
        hidden = max(256, d_model // 2)
        self.proj = torch.nn.Sequential(torch.nn.Linear(d_model, hidden), torch.nn.GELU(), torch.nn.Dropout(0.1))
        self.temporal = torch.nn.Sequential(torch.nn.Conv1d(hidden, hidden, kernel_size=7, padding=3),
                                            torch.nn.GELU(), torch.nn.Dropout(0.1))
        self.out = torch.nn.Linear(hidden, 1)

    def forward(self, input_features):
        h = self.proj(self.whisper.encoder(input_features=input_features).last_hidden_state)
        h = self.temporal(h.permute(0, 2, 1)).permute(0, 2, 1)
        return self.out(h).squeeze(-1)


locator = WhisperSegmenter()
locator.load_state_dict(torch.load(hf_hub_download("laion/vocalburst-locator", "model_v2.pt"), map_location="cpu"),
                        strict=True)
locator = locator.eval().to(dev)
whisper_fe = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
ANALYSIS_VERSION = 2          # analysis.json records of an older version are analysed again

# --- text -------------------------------------------------------------------------------------------------------
FILLER = re.compile(r"(?:h+m+|x+m+|m{2,}|m+h+m+|a+(?:-a+)+|a{3,}|e+(?:-e+)*h*|u+h+m*|u+m+|u{3,}|a+ha+)", re.I)
PUNCT = ".,!?…:;\"«»“”()[]—–-"


def is_filler(tok):
    core = tok.strip(PUNCT)
    return bool(core) and FILLER.fullmatch(core) is not None


def only_fillers(text):
    toks = [t for t in text.split() if t.strip(PUNCT)]
    return bool(toks) and all(is_filler(t) for t in toks)


def drop_fillers(text):
    """The text without its filler words, with the punctuation around them tidied."""
    out, cap = [], False
    for t in text.split():
        if not is_filler(t):
            if cap:                                       # "u. A-a, biz" -> "u. Biz"
                t, cap = t[:1].upper() + t[1:], False
            out.append(t)
        elif (not out or re.search(r"[.!?…]$", out[-1])) and t.strip(PUNCT)[:1].isupper():
            cap = True
        elif out and (end := re.search(r"[.!?…]+$", t)) and not re.search(r"[.!?…]$", out[-1]):
            out[-1] = re.sub(r"[,;:]+$", "", out[-1]) + end.group().replace("?", ".")   # "Shunaqa, mhm." -> "Shunaqa."
    s = " ".join(out)
    s = re.sub(r"\s+([,.!?…:;])", r"\1", s)
    s = re.sub(r"([,;:])(?:\s*[,;:])+", r"\1", s)        # ", ," left where a filler was between commas
    s = re.sub(r"[,;:]\s*([.!?…])", r"\1", s)
    s = re.sub(r"^[\s,.!?…:;—–-]+", "", s).strip()
    return s[:1].upper() + s[1:] if s and text.lstrip()[:1].isupper() else s


ONES = ["nol", "bir", "ikki", "uch", "to'rt", "besh", "olti", "yetti", "sakkiz", "to'qqiz"]
TENS = ["", "o'n", "yigirma", "o'ttiz", "qirq", "ellik", "oltmish", "yetmish", "sakson", "to'qson"]


def number_to_uz(n):
    if n < 10:
        return ONES[n]
    if n < 100:
        return " ".join(w for w in (TENS[n // 10], ONES[n % 10] if n % 10 else "") if w)
    if n < 1000:
        return ("yuz" if n // 100 == 1 else f"{ONES[n // 100]} yuz") + (f" {number_to_uz(n % 100)}" if n % 100 else "")
    for size, name in [(10**9, "milliard"), (10**6, "million"), (1000, "ming")]:
        if n >= size:
            head = name if (n // size == 1 and size == 1000) else f"{number_to_uz(n // size)} {name}"
            return head + (f" {number_to_uz(n % size)}" if n % size else "")


def spoken(word):
    """A transcript word as the letters that are said: digits as Uzbek words, no punctuation. '' = nothing said."""
    w = re.sub(r"[‘’ʻʼ`]", "'", word.lower()).replace("%", " foiz")
    w = re.sub(r"\d+", lambda m: number_to_uz(int(m.group())) if len(m.group()) <= 12
               else " ".join(ONES[int(d)] for d in m.group()), w)
    return "".join(ch for ch in w if ch in LETTERS)


# --- audio ------------------------------------------------------------------------------------------------------
def frame_db(x16, n):
    x16 = np.pad(x16, (0, max(0, n * HOP - len(x16))))[:n * HOP]
    return 10 * np.log10(np.maximum((x16.reshape(n, HOP).astype(np.float64) ** 2).mean(1), 1e-12))


def runs(mask, min_len, max_hole=2):
    """[start, end) of the True runs of mask, with holes of up to max_hole frames filled in."""
    idx = np.flatnonzero(mask)
    if not len(idx):
        return []
    out, a, b = [], idx[0], idx[0] + 1
    for i in idx[1:]:
        if i - b <= max_hole:
            b = i + 1
        else:
            out.append((a, b))
            a, b = i, i + 1
    out.append((a, b))
    return [(a, b) for a, b in out if b - a >= min_len]


@torch.inference_mode()
def emissions(x16):
    iv = processor.feature_extractor(x16, sampling_rate=16000, return_tensors="pt").input_values.to(dev)
    return torch.log_softmax(mms(iv).logits.float(), -1)[0].cpu()


INV = {i: ch for ch, i in VOCAB.items()}


def heard(em, a, b):
    """What MMS hears in frames a..b by itself (greedy CTC): '' when it hears no letters."""
    out, prev = [], None
    for i in em[a:b].argmax(-1).tolist():
        if i != prev and i != BLANK:
            out.append(INV[i])
        prev = i
    return "".join(out).replace("|", " ").strip()


def align(em, units):
    """Start / end frame and mean probability of every unit (word) via CTC forced alignment."""
    toks = torch.tensor([[VOCAB[ch] for u in units for ch in u]], dtype=torch.int32)
    al, sc = AF.forced_align(em[None], toks, blank=BLANK)
    spans = AF.merge_tokens(al[0], sc[0].exp())
    out, k = [], 0
    for u in units:
        sp = spans[k:k + len(u)]
        k += len(u)
        out.append((sp[0].start, sp[-1].end))
    return out, float(sc[0].exp().mean())


@torch.inference_mode()
def burst_probs(x16, n):
    """The locator's burst probability for each of n 20 ms frames (one 30 s window: the chunks are shorter)."""
    feats = whisper_fe(x16[:480000], sampling_rate=16000, return_tensors="pt").input_features.to(dev)
    p = torch.sigmoid(locator(feats))[0].float().cpu().numpy()
    return np.pad(p, (0, max(0, n - len(p))))[:n]


@torch.inference_mode()
def ast_groups(segs16):
    feats = ast_fe(segs16, sampling_rate=16000, return_tensors="pt")["input_values"].to(dev)
    p = torch.sigmoid(ast(feats).logits).cpu().numpy()
    return [{g: round(float(row[ids].max()), 3) for g, ids in GROUP_IDS.items()} for row in p]


def voicing(seg16):
    """Share of voiced frames, number of voiced stretches, and pitch spread in semitones (pYIN)."""
    seg16 = np.pad(seg16, (0, max(0, 1600 - len(seg16))))
    f0, vflag, _ = librosa.pyin(seg16, fmin=65, fmax=450, sr=16000, frame_length=640, hop_length=160)
    segs = sum(1 for a, b in runs(vflag, 3, 1))
    st = float(np.std(12 * np.log2(f0[vflag]))) if vflag.sum() >= 3 else None
    return float(vflag.mean()), segs, st


def analyse(c):
    x, sr = sf.read(ds / c["file"], dtype="float32")
    x16 = AF.resample(torch.from_numpy(x), sr, 16000).numpy()
    toks = c["text"].split()
    units = [spoken(t) for t in toks]
    idx = [i for i, u in enumerate(units) if u]
    rec = dict(duration=c["duration"])
    if not idx:
        return dict(rec, error="no letters in the text")
    em = emissions(x16)
    n = em.shape[0]
    try:
        spans, prob = align(em, [units[i] for i in idx])
    except Exception as e:
        return dict(rec, error=f"alignment failed: {str(e)[:120]}")
    word = dict(zip(idx, spans))
    fdb = frame_db(x16, n)
    sound = fdb >= np.percentile(fdb, 90) - SOUND_DB
    p0, p1 = round(WORD_PAD[0] / FRAME), round(WORD_PAD[1] / FRAME)
    fill = {i for i in idx if is_filler(toks[i])}
    real = [i for i in idx if i not in fill]
    covered = np.zeros(n, bool)
    for i in real:
        s, e = word[i]
        covered[max(0, s - p0):min(n, e + p1)] = True
    on_words = covered.copy()
    bp = burst_probs(x16, n)
    burst = np.zeros(n, bool)
    for a, b in runs(bp >= BURST_THRESHOLD, 5, 5):   # the locator's own post-processing: >= 0.1 s, gaps <= 0.1 s merged
        burst[a:b] = True
    written = []
    for i in sorted(fill):   # a written filler = the sound between its neighbouring words
        s, e = word[i]
        prev = max((word[j][1] for j in real if j < i), default=None)
        nxt = min((word[j][0] for j in real if j > i), default=None)
        lo = min(s, prev + p1 if prev is not None else 0)
        hi = max(e, nxt - p0 if nxt is not None else n)
        snd = np.flatnonzero(sound[lo:hi])
        a, b = (min(s, lo + snd[0]), max(e, lo + snd[-1] + 1)) if len(snd) else (s, e)
        written.append([round(a * FRAME, 3), round(b * FRAME, 3), toks[i]])
        covered[a:b] = True
    gaps = []
    for a, b in runs((sound | burst) & ~covered, round(MIN_GAP_SEC / FRAME)):   # a burst counts even when quiet
        voiced, segs, st = voicing(x16[a * HOP:b * HOP])
        gaps.append(dict(s=round(a * FRAME, 3), e=round(b * FRAME, 3), heard=heard(em, a, b),
                         burst=round(float(bp[a:b].mean()), 3),
                         voiced=round(voiced, 3), segs=segs, pitch_st=None if st is None else round(st, 2),
                         touches=bool((a > 0 and covered[a - 1]) or (b < n and covered[b]))))
    if gaps:
        for g, probs in zip(gaps, ast_groups([x16[int(g["s"] * 16000):int(g["e"] * 16000)] for g in gaps])):
            g.update(probs)
    return dict(rec, v=ANALYSIS_VERSION, prob=round(prob, 3), words=len(real), written=written, gaps=gaps,
                burst_in_words=round(float((burst & on_words).sum() * FRAME), 2))


def gap_type(g):
    group, p = max(((k, g[k]) for k in GROUPS), key=lambda kv: kv[1])
    if g["burst"] >= BURST_MIN:                       # the locator hears a vocal burst; AST names it when it can
        return group if p >= AST_MIN else "burst"
    if p >= AST_MIN:
        return group
    h = re.sub(r"[^a-z']", "", g["heard"])
    if len(h) >= MIN_HEARD_LETTERS:                  # MMS hears letters here by itself
        return "filler" if FILLER.fullmatch(h) else "speech"
    if g["voiced"] < 0.3:
        return "breath"
    if g["voiced"] >= 0.6 and g["segs"] <= 1 and g["pitch_st"] is not None and g["pitch_st"] <= MAX_PITCH_ST:
        return "filler"                               # one steady voiced sound: ee, mm, a-a, a drawn-out vowel
    return "word_edge" if g["touches"] and g["e"] - g["s"] < 0.4 else "other"


def room_tone(x, n):
    """The quietest n samples of x (the chunk's own background), for the pauses left by cuts."""
    if len(x) <= n:
        return np.zeros(n, np.float32)
    hop = max(1, n // 4)
    starts = range(0, len(x) - n, hop)
    s = min(starts, key=lambda i: float(np.mean(x[i:i + n] ** 2)))
    return x[s:s + n].copy()


def splice(x, cuts):
    """x without the [start, end] seconds in cuts; a cut inside x leaves a short pause of room tone."""
    regions = []
    for s, e in sorted((max(0.0, s), min(len(x) / SR, e)) for s, e, *_ in cuts):
        if regions and s <= regions[-1][1]:
            regions[-1][1] = max(regions[-1][1], e)
        elif e > s:
            regions.append([s, e])
    tone = room_tone(x, int(PAUSE_SEC[1] * SR))
    fade = int(FADE_SEC * SR)
    pieces, pos = [], 0
    for s, e in regions:
        a, b = int(s * SR), int(e * SR)
        pieces.append(x[pos:a])
        if a > 0 and b < len(x):
            pieces.append(tone[:int(np.clip(0.4 * (e - s), *PAUSE_SEC) * SR)])
        pos = b
    pieces.append(x[pos:])
    pieces = [p.astype(np.float32).copy() for p in pieces if len(p)]
    for k, p in enumerate(pieces):
        f = min(fade, len(p) // 2)
        if f and k > 0:
            p[:f] *= np.linspace(0, 1, f, dtype=np.float32)
        if f and k < len(pieces) - 1:
            p[-f:] *= np.linspace(1, 0, f, dtype=np.float32)
    return np.concatenate(pieces) if pieces else np.zeros(0, np.float32)


def save_json(path, obj, indent=1):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    os.replace(tmp, path)


def write_wav(path, y):
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.abs(y).max()) if len(y) else 0.0
    sf.write(path, y * (0.999 / peak) if peak > 0.999 else y, SR, subtype="PCM_16")


print(f"{DATASET}: {len(chunks)} chunks, {sum(c['duration'] for c in chunks) / 3600:.2f} h; device {dev}; "
      f"MMS letters: {''.join(sorted(LETTERS))}")

/home/intern/projects/tts/runs/test/dataset_clean: 412 chunks, 1.32 h; device cuda; MMS letters: 'abcdefghijklmnopqrstuvwxyz


## 1. Analyse

In [4]:
analysis_path = work / "analysis.json"
analysis = json.load(open(analysis_path, encoding="utf-8")) if analysis_path.exists() else {}
scope = [c for c in chunks if SCOPE == "all" or any(is_filler(t) for t in c["text"].split())]
scope = scope[:LIMIT] if LIMIT else scope
def stale(c):
    """Not analysed yet, analysed for another version of the chunk, or by an older version of this notebook."""
    r = analysis.get(c["file"])
    return r is None or r.get("duration") != c["duration"] or (not r.get("error") and r.get("v") != ANALYSIS_VERSION)


todo = [c for c in scope if stale(c)]
print(f"scope {SCOPE}: {len(scope)} chunks; {len(scope) - len(todo)} already analysed, {len(todo)} to go "
      f"({sum(c['duration'] for c in todo) / 3600:.1f} h of audio)")
t0 = time.time()
try:
    for k, c in enumerate(tqdm(todo, desc="analyse"), 1):
        try:
            analysis[c["file"]] = analyse(c)
        except Exception as e:
            analysis[c["file"]] = dict(duration=c["duration"], error=f"{type(e).__name__}: {str(e)[:120]}")
        if k % SAVE_EVERY == 0:
            save_json(analysis_path, analysis)
finally:
    save_json(analysis_path, analysis)
errors = Counter(analysis[c["file"]]["error"].split(":")[0] for c in scope if analysis[c["file"]].get("error"))
print(f"done in {(time.time() - t0) / 60:.1f} min; errors: {dict(errors) or 'none'}")

scope all: 412 chunks; 0 already analysed, 412 to go (1.3 h of audio)
done in 5.2 min; errors: none


## 2. Decide

In [5]:
decision = {}   # file -> dict(action="keep" | "edit" | "drop", reason, cuts, text)
for c in chunks:
    f, r = c["file"], analysis.get(c["file"])
    if only_fillers(c["text"]):
        decision[f] = dict(action="drop", reason="filler_only")
    elif stale(c):
        decision[f] = dict(action="keep", reason="not analysed")
    elif r.get("error"):
        decision[f] = dict(action="drop", reason="align_failed")
    elif MIN_ALIGN_PROB is not None and r["prob"] < MIN_ALIGN_PROB:
        decision[f] = dict(action="drop", reason="bad_alignment")
    elif MAX_BURST_IN_WORDS_SEC is not None and r["burst_in_words"] >= MAX_BURST_IN_WORDS_SEC:
        decision[f] = dict(action="drop", reason="burst_over_words")
    else:
        typed = [(g, gap_type(g)) for g in r["gaps"]]
        untranscribed = sum(g["e"] - g["s"] for g, t in typed if t == "speech")
        cuts = [(s, e, "written") for s, e, _ in r["written"]] + [(g["s"], g["e"], t) for g, t in typed if t in CUT_TYPES]
        text = drop_fillers(c["text"]) if r["written"] else c["text"]
        if untranscribed >= MIN_UNTRANSCRIBED_SEC:
            decision[f] = dict(action="drop", reason="untranscribed_speech", types=[t for _, t in typed])
        elif cuts:
            removed = sum(e - s for s, e, _ in cuts)
            decision[f] = dict(action="edit", cuts=cuts, text=text, types=[t for _, t in typed])
            if not text or c["duration"] - removed < MIN_SEC:
                decision[f] = dict(action="drop", reason="too_short_after_cut")
        else:
            decision[f] = dict(action="keep", types=[t for _, t in typed])

act = Counter(d["action"] for d in decision.values())
hours = lambda fs: sum(by_file[f]["duration"] for f in fs) / 3600
print(f"{'':6} {'chunks':>7} {'hours':>7}")
for a in ["keep", "edit", "drop"]:
    fs = [f for f, d in decision.items() if d["action"] == a]
    print(f"{a:6} {len(fs):7d} {hours(fs):7.2f}")
n_na = sum(1 for d in decision.values() if d.get("reason") == "not analysed")
if n_na:
    print(f"({n_na} chunks are not analysed yet and are kept as they are: SCOPE = {SCOPE!r}, or run step 1 again)")
print("\ndropped:", dict(Counter(d["reason"] for d in decision.values() if d["action"] == "drop")))
types = Counter(t for d in decision.values() for t in d.get("types", []))
print("unwritten sounds found:", dict(types))
cut = Counter(t for d in decision.values() if d["action"] == "edit" for *_, t in d["cuts"])
cut_sec = defaultdict(float)
for d in decision.values():
    if d["action"] == "edit":
        for s, e, t in d["cuts"]:
            cut_sec[t] += e - s
print("cut from the audio:", {t: f"{n} ({cut_sec[t] / 60:.1f} min)" for t, n in cut.items()})
print("texts changed:", sum(1 for f, d in decision.items() if d["action"] == "edit" and d["text"] != by_file[f]["text"]))
biw = np.array([r["burst_in_words"] for r in analysis.values() if "burst_in_words" in r])
if len(biw):
    print(f"chunks with a vocal burst on top of words: {(biw > 0).sum()} "
          f"(>= 0.5 s: {(biw >= 0.5).sum()}; drop them with MAX_BURST_IN_WORDS_SEC)")
probs = np.array([r["prob"] for r in analysis.values() if "prob" in r])
if len(probs):
    print("alignment probability, percentiles 1/5/50:", np.round(np.percentile(probs, [1, 5, 50]), 3))
save_json(work / "report.json", {f: dict(d, text_before=by_file[f]["text"]) for f, d in decision.items()
                                 if d["action"] != "keep" or d.get("types")})

        chunks   hours
keep       292    0.63
edit       118    0.68
drop         2    0.01

dropped: {'untranscribed_speech': 1, 'too_short_after_cut': 1}
unwritten sounds found: {'breath': 682, 'filler': 97, 'other': 56, 'word_edge': 68, 'speech': 6, 'burst': 12, 'cough': 3, 'laugh': 2}
cut from the audio: {'filler': '94 (0.8 min)', 'other': '55 (0.5 min)', 'speech': '5 (0.0 min)', 'burst': '12 (0.1 min)', 'written': '23 (0.4 min)', 'cough': '3 (0.0 min)', 'laugh': '2 (0.0 min)'}
texts changed: 21
chunks with a vocal burst on top of words: 41 (>= 0.5 s: 5; drop them with MAX_BURST_IN_WORDS_SEC)
alignment probability, percentiles 1/5/50: [0.456 0.793 0.897]


## 3. Review
Writes `outputs/fillers/review/<kind>/<n>_<chunk>/` with `before.wav`, `after.wav` (when edited), `sound.wav` (the detected sound with 0.3 s around it), `info.json`; and `index.html` to listen in a browser. Every dropped chunk is linked into `outputs/fillers/dropped/<reason>/`.

In [6]:
review = work / "review"
shutil.rmtree(review, ignore_errors=True)
kinds = defaultdict(list)   # kind -> [(file, the gap or cut to show)]
for f, d in decision.items():
    if d["action"] == "edit":
        for s, e, t in d["cuts"]:
            kinds["written" if t == "written" else f"cut_{t}"].append((f, (s, e)))
    elif d["action"] == "drop":
        kinds[f"drop_{d['reason']}"].append((f, None))
    r = analysis.get(f) or {}
    for g in r.get("gaps", []):
        t = gap_type(g)
        if t not in CUT_TYPES:
            kinds[f"kept_{t}"].append((f, (g["s"], g["e"])))
rows = []
for kind, items in sorted(kinds.items()):
    seen, picks = set(), []
    for f, span in items[::max(1, len(items) // N_REVIEW)]:
        if f not in seen and len(picks) < N_REVIEW:
            seen.add(f)
            picks.append((f, span))
    for k, (f, span) in enumerate(picks, 1):
        c, d = by_file[f], decision[f]
        folder = review / kind / f"{k:02d}_{Path(f).stem}"
        x, _ = sf.read(ds / f, dtype="float32")
        write_wav(folder / "before.wav", x)
        if d["action"] == "edit":
            write_wav(folder / "after.wav", splice(x, d["cuts"]))
        if span:
            write_wav(folder / "sound.wav", x[max(0, int((span[0] - 0.3) * SR)):int((span[1] + 0.3) * SR)])
        info = dict(kind=kind, file=f, span=span, text=c["text"], new_text=d.get("text"), action=d["action"])
        save_json(folder / "info.json", info)
        rows.append(dict(info, folder=folder.relative_to(review).as_posix()))
dropped = work / "dropped"
shutil.rmtree(dropped, ignore_errors=True)
for f, d in decision.items():
    if d["action"] == "drop":
        (dropped / d["reason"]).mkdir(parents=True, exist_ok=True)
        try:
            os.link(ds / f, dropped / d["reason"] / Path(f).name)   # a hard link: no extra space
        except OSError:
            shutil.copy2(ds / f, dropped / d["reason"] / Path(f).name)

player = lambda r, n: (f"<b>{n}</b><br><audio controls preload='none' src='{r['folder']}/{n}.wav'></audio>"
                       if (review / r["folder"] / f"{n}.wav").exists() else "")
body = "".join(
    f"<tr><td colspan=3><h3>{r['kind']}: {html.escape(r['file'])}"
    + (f" · {r['span'][0]:.2f}-{r['span'][1]:.2f} s" if r["span"] else "") + "</h3>"
    f"<p>{html.escape(r['text'])}</p>"
    + (f"<p><i>new text:</i> {html.escape(r['new_text'])}</p>" if r["new_text"] and r["new_text"] != r["text"] else "")
    + f"</td></tr><tr><td>{player(r, 'sound')}</td><td>{player(r, 'before')}</td><td>{player(r, 'after')}</td></tr>"
    for r in rows)
(review / "index.html").write_text(
    "<!DOCTYPE html><meta charset='utf-8'><title>Filler review</title>"
    "<style>body{font-family:system-ui;margin:24px}td{vertical-align:top;padding:4px 16px 12px 0}"
    "h3{margin:20px 0 4px}p{margin:0 0 4px;color:#444}</style><h1>Filler review</h1>"
    "<p>sound = the detected sound with 0.3 s around it; after = the chunk as it goes into the dataset.</p>"
    f"<table>{body}</table>", encoding="utf-8")
print(f"{len(rows)} examples in {review}; {sum(1 for d in decision.values() if d['action'] == 'drop')} dropped chunks "
      f"in {dropped}")
for kind, items in sorted(kinds.items()):
    print(f"  {kind:28} {len(items):6d}")

60 examples in /home/intern/projects/tts/runs/test/outputs/fillers/review; 2 dropped chunks in /home/intern/projects/tts/runs/test/outputs/fillers/dropped
  cut_burst                        12
  cut_cough                         3
  cut_filler                       94
  cut_laugh                         2
  cut_other                        55
  cut_speech                        5
  drop_too_short_after_cut          1
  drop_untranscribed_speech         1
  kept_breath                     682
  kept_word_edge                   68
  written                          23


In [7]:
# listen to a few here (each player adds ~1 MB to the notebook; index.html has them all)
from IPython.display import Audio, display

for r in [r for r in rows if r["kind"] in ("written", "cut_filler", "cut_burst", "cut_cough", "cut_speech")][:5]:
    print(f"\n{r['kind']}: {r['file']} | {r['text']}" + (f"\n  -> {r['new_text']}" if r["new_text"] else ""))
    for n in ("sound", "before", "after"):
        p = review / r["folder"] / f"{n}.wav"
        if p.exists():
            print(n)
            display(Audio(str(p)))


cut_burst: chunks/001_0012_speaker_0.wav | Fikringiz bilan o'zingizni davolang degan narsaga men aslida juda ham qarshiman. Lekin ishonchim komil, sizda ham shunaqa inson bo'lgan. Menda aniq bo'lgan. Meni rahmatli ammam joylari, jannatda bo'lsin, saraton kasalligi bilan ularni diagnoz qo'yilganda, uylariga ko'rgani borganimizda hali ham esimda. Men kasal emasman, men tuzalaman, men sog'lomman degan ta'kidlarni o'zlariga aytib yurardilar. Va bu kasallik ularni tark etgan.
  -> Fikringiz bilan o'zingizni davolang degan narsaga men aslida juda ham qarshiman. Lekin ishonchim komil, sizda ham shunaqa inson bo'lgan. Menda aniq bo'lgan. Meni rahmatli ammam joylari, jannatda bo'lsin, saraton kasalligi bilan ularni diagnoz qo'yilganda, uylariga ko'rgani borganimizda hali ham esimda. Men kasal emasman, men tuzalaman, men sog'lomman degan ta'kidlarni o'zlariga aytib yurardilar. Va bu kasallik ularni tark etgan.
sound
<IPython.lib.display.Audio object>
before
<IPython.lib.display.Audio object>
af

## 4. Build `dataset_nofillers/`

In [8]:
tmp = out_dir.with_name(out_dir.name + ".tmp")
shutil.rmtree(tmp, ignore_errors=True)
(tmp / "chunks").mkdir(parents=True)
kept = []
for c in tqdm(chunks, desc="build"):
    f, d = c["file"], decision[c["file"]]
    if d["action"] == "drop":
        continue
    if d["action"] == "keep":
        try:
            os.link(ds / f, tmp / f)          # unchanged: a hard link, no extra space
        except OSError:
            shutil.copy2(ds / f, tmp / f)
        kept.append(c)
        continue
    x, _ = sf.read(ds / f, dtype="float32")
    y = splice(x, d["cuts"])
    write_wav(tmp / f, y)
    kept.append(dict(c, text=d["text"], duration=round(len(y) / SR, 3), fillers=dict(
        cuts=[[round(s, 3), round(e, 3), t] for s, e, t in d["cuts"]], text_before=c["text"])))
save_json(tmp / "metadata.json", dict(meta, chunks=kept, fillers=dict(
    SOUND_DB=SOUND_DB, WORD_PAD=WORD_PAD, MIN_GAP_SEC=MIN_GAP_SEC, AST_MIN=AST_MIN, MAX_PITCH_ST=MAX_PITCH_ST,
    MIN_HEARD_LETTERS=MIN_HEARD_LETTERS, BURST_THRESHOLD=BURST_THRESHOLD,
    BURST_MIN=BURST_MIN, MAX_BURST_IN_WORDS_SEC=MAX_BURST_IN_WORDS_SEC, CUT_TYPES=sorted(CUT_TYPES), MIN_UNTRANSCRIBED_SEC=MIN_UNTRANSCRIBED_SEC, MIN_ALIGN_PROB=MIN_ALIGN_PROB,
    MIN_SEC=MIN_SEC, SCOPE=SCOPE)), indent=2)
if out_dir.exists():
    shutil.rmtree(out_dir)                    # the previous build: hard links and edited copies only
os.replace(tmp, out_dir)
print(f"{OUT}: {len(kept)} chunks, {sum(c['duration'] for c in kept) / 3600:.2f} h "
      f"(from {len(chunks)} chunks, {sum(c['duration'] for c in chunks) / 3600:.2f} h)")

# check
m2 = json.load(open(out_dir / "metadata.json", encoding="utf-8"))
wavs = {p.name for p in (out_dir / "chunks").glob("*.wav")}
names = {Path(c["file"]).name for c in m2["chunks"]}
bad = [c["file"] for c in m2["chunks"] if abs(sf.info(out_dir / c["file"]).frames / SR - c["duration"]) > 0.002]
print("every chunk has its wav:", names == wavs, "| durations match the wavs:", "all" if not bad else f"NO: {bad[:5]}")
edited = [c for c in m2["chunks"] if "fillers" in c]
print(f"edited: {len(edited)}, of them text changed: {sum(1 for c in edited if c['text'] != c['fillers']['text_before'])}")
for c in [c for c in edited if c["text"] != c["fillers"]["text_before"]][:4]:
    print(f"  {c['fillers']['text_before'][:70]}\n   -> {c['text'][:70]}")

/home/intern/projects/tts/runs/test/dataset_nofillers: 410 chunks, 1.30 h (from 412 chunks, 1.32 h)
every chunk has its wav: True | durations match the wavs: all
edited: 118, of them text changed: 21
  Xotirjam bo'lib turib, bo'ladi Xudo xohlasa degan narsa bilan kiradi. 
   -> Xotirjam bo'lib turib, bo'ladi Xudo xohlasa degan narsa bilan kiradi. 
  10% yoki 15% holatda foydalanamiz. Undan tashqari, undan boshqa vaqtla
   -> 10% yoki 15% holatda foydalanamiz. Undan tashqari, undan boshqa vaqtla
  Oson, balki aybdor o'zingdirsan, balki sen bir nima qilarsan, balki ha
   -> Oson, balki aybdor o'zingdirsan, balki sen bir nima qilarsan, balki ha
  narsani o'zim qilib ko'rganman, endi qilmagan narsamni maslahat bera o
   -> narsani o'zim qilib ko'rganman, endi qilmagan narsamni maslahat bera o


## 5. Sentence chunks → `dataset_final/`
Run 5a once: it aligns every chunk (about 2-3 min per hour of audio on the shared GPU) and saves it. 5b and 5c are quick: change the step 5 settings in the config cell and run them again.

In [9]:
# 5a. word times: forced alignment of every dataset_nofillers/ chunk (its final audio and text), saved in words.json
words_path = work / "words.json"
words = json.load(open(words_path, encoding="utf-8")) if words_path.exists() else {}
nf_meta = json.load(open(out_dir / "metadata.json", encoding="utf-8"))
nf_chunks = nf_meta["chunks"]
final_dir = root / FINAL


def word_times(c):
    """[first token, end token, start s, end s] of every aligned word of chunk c. A word owns the tokens without
    letters that follow it (a lone "—"); the first word also those before it."""
    rec = dict(duration=c["duration"], text=c["text"])
    toks = c["text"].split()
    idx = [i for i, t in enumerate(toks) if spoken(t)]
    if not idx:
        return dict(rec, error="no letters in the text")
    x, sr = sf.read(out_dir / c["file"], dtype="float32")
    try:
        em = emissions(AF.resample(torch.from_numpy(x), sr, 16000).numpy())
        spans, _ = align(em, [spoken(toks[i]) for i in idx])
    except Exception as e:
        return dict(rec, error=f"alignment failed: {str(e)[:120]}")
    ends = idx[1:] + [len(toks)]
    return dict(rec, words=[[0 if k == 0 else i, ends[k], round(s * FRAME, 3), round(e * FRAME, 3)]
                            for k, (i, (s, e)) in enumerate(zip(idx, spans))])


todo = [c for c in nf_chunks if (words.get(c["file"]) or {}).get("text") != c["text"]
        or words[c["file"]].get("duration") != c["duration"]]
print(f"{OUT}: {len(nf_chunks)} chunks; {len(nf_chunks) - len(todo)} aligned before, {len(todo)} to align "
      f"({sum(c['duration'] for c in todo) / 3600:.2f} h of audio)")
t0 = time.time()
try:
    for k, c in enumerate(tqdm(todo, desc="align"), 1):
        words[c["file"]] = word_times(c)
        if k % SAVE_EVERY == 0:
            save_json(words_path, words)
finally:
    save_json(words_path, words)
errors = Counter(words[c["file"]]["error"].split(":")[0] for c in nf_chunks if words[c["file"]].get("error"))
print(f"done in {(time.time() - t0) / 60:.1f} min; not aligned (they stay as they are): {dict(errors) or 'none'}")


/home/intern/projects/tts/runs/test/dataset_nofillers: 410 chunks; 0 aligned before, 410 to align (1.30 h of audio)
done in 2.8 min; not aligned (they stay as they are): none


In [10]:
# 5b. join consecutive chunks of a speaker, cut them at sentence ends, build dataset_final/
SENT_END = re.compile(r"[.!?…]+['\"»”)\]]*$")   # a sentence end, maybe followed by closing quotes or brackets
p0, p1 = WORD_PAD


def join_groups(chunks):
    """Lists of chunks to join: same speaker, each starting where the one before ends in the original audio,
    denoised the same way, all aligned. Every other chunk is a list of its own."""
    out, by_pod = [], defaultdict(list)
    for c in chunks:
        by_pod[c["speaker"][:3]].append(c)
    for cs in by_pod.values():
        for c in sorted(cs, key=lambda c: c["start"]):
            prev = out[-1][-1] if out else None
            if (prev is not None and prev["speaker"] == c["speaker"] and abs(c["start"] - prev["end"]) < 0.01
                    and prev.get("denoised", "") == c.get("denoised", "")
                    and "words" in words[prev["file"]] and "words" in words[c["file"]]):
                out[-1].append(c)
            else:
                out.append([c])
    return out


def between(cut, n):
    """(first word, last word) of every piece between the cuts; a cut j lies between word j and word j + 1."""
    b = sorted(cut)
    return list(zip([0] + [j + 1 for j in b], b + [n - 1]))


def plan(g):
    """The pieces of group g on its clock (its chunks end to end): dict(t0, t1, text, cut, short). Also returns the
    clock offsets of its chunks and the pauses after its sentence ends."""
    offs = np.cumsum([0.0] + [c["duration"] for c in g])
    toks = [c["text"].split() for c in g]
    W = [(k, i0, i1, offs[k] + s, offs[k] + e) for k, c in enumerate(g) for i0, i1, s, e in words[c["file"]]["words"]]
    n = len(W)
    sent = [bool(SENT_END.search(toks[k][i1 - 1])) for k, _, i1, _, _ in W]
    lo = [W[j][4] + p1 for j in range(n - 1)]                       # padded end of word j
    hi = [W[j + 1][3] - p0 for j in range(n - 1)]                   # padded start of word j + 1
    pause = [max(0.0, b - a) for a, b in zip(lo, hi)]
    joint = [W[j][0] != W[j + 1][0] for j in range(n - 1)]          # the old cut between two chunks
    mid = [offs[W[j + 1][0]] if joint[j] else (lo[j] + hi[j]) / 2 for j in range(n - 1)]

    def t_start(a):
        return max(0.0, W[0][3] - p0 - EDGE_SILENCE_SEC) if a == 0 else max(mid[a - 1], hi[a - 1] - EDGE_SILENCE_SEC)

    def t_end(z):
        return min(offs[-1], W[-1][4] + p1 + EDGE_SILENCE_SEC) if z == n - 1 else min(mid[z], lo[z] + EDGE_SILENCE_SEC)

    def length(a, z):
        return t_end(z) - t_start(a)

    cut = {j: "pause" for j in range(n - 1) if sent[j] and pause[j] >= SPLIT_PAUSE_SEC}

    def split_long(a, z):
        if length(a, z) <= MAX_CHUNK_SEC:
            return
        cand = [j for j in range(a, z) if sent[j] and pause[j] >= MIN_CUT_PAUSE_SEC
                and min(length(a, j), length(j + 1, z)) >= MIN_CHUNK_SEC]
        if cand:
            j = max(cand, key=lambda j: (round(pause[j], 1), -abs(length(a, j) - length(j + 1, z))))
            cut[j] = "long"
            split_long(a, j)
            split_long(j + 1, z)
        else:                                       # no sentence end to cut at: the old cuts stay
            for j in range(a, z):
                if joint[j]:
                    cut[j] = "old"

    for a, z in between(cut, n):
        split_long(a, z)
    while True:                                     # a short piece joins its neighbour across the shorter pause
        segs = between(cut, n)
        fix = None
        for i, (a, z) in enumerate(segs):
            if length(a, z) >= MIN_CHUNK_SEC:
                continue
            sides = ([a - 1] if i > 0 and length(segs[i - 1][0], z) <= MAX_CHUNK_SEC else []) + \
                    ([z] if i < len(segs) - 1 and length(a, segs[i + 1][1]) <= MAX_CHUNK_SEC else [])
            if sides:
                fix = min(sides, key=lambda j: pause[j])
                break
        if fix is None:
            break
        del cut[fix]
    pieces = [dict(t0=t_start(a), t1=t_end(z), cut=cut.get(a - 1, "start"), short=length(a, z) < MIN_CHUNK_SEC,
                   text=" ".join(t for k, i0, i1, *_ in W[a:z + 1] for t in toks[k][i0:i1])) for a, z in between(cut, n)]
    return offs, pieces, [pause[j] for j in range(n - 1) if sent[j]], sum(joint[j] and j not in cut for j in range(n - 1))


def piece_audio(xs, offs, t0, t1):
    """Seconds t0..t1 of a group's clock: the slices of its chunks end to end, with a short fade at both ends."""
    parts = [x[int(round((max(t0, offs[k]) - offs[k]) * SR)):int(round((min(t1, offs[k + 1]) - offs[k]) * SR))]
             for k, x in enumerate(xs) if min(t1, offs[k + 1]) > max(t0, offs[k])]
    y = np.concatenate(parts).astype(np.float32)
    f = min(int(FADE_SEC * SR), len(y) // 2)
    if f:
        y[:f] *= np.linspace(0, 1, f, dtype=np.float32)
        y[-f:] *= np.linspace(1, 0, f, dtype=np.float32)
    return y


sgroups = join_groups(nf_chunks)
tmp = final_dir.with_name(final_dir.name + ".tmp")
shutil.rmtree(tmp, ignore_errors=True)
(tmp / "chunks").mkdir(parents=True)
final_rows, made_from, st = [], {}, Counter()
sent_pauses, short_sec = [], 0.0
for g in tqdm(sgroups, desc="sentences"):
    if "words" not in words[g[0]["file"]]:          # not aligned: kept as it is
        c = g[0]
        try:
            os.link(out_dir / c["file"], tmp / c["file"])
        except OSError:
            shutil.copy2(out_dir / c["file"], tmp / c["file"])
        final_rows.append(dict(c, sources=[[c["file"], 0.0, c["duration"]]]))
        st["kept, not aligned"] += 1
        continue
    offs, pieces, sp, n_joined = plan(g)
    assert " ".join(p["text"] for p in pieces) == " ".join(t for c in g for t in c["text"].split()), g[0]["file"]
    sent_pauses += sp
    st["old cuts joined"] += n_joined
    xs = [sf.read(out_dir / c["file"], dtype="float32")[0] for c in g]
    for i, p in enumerate(pieces):
        if p["short"]:
            st["pieces left out (short)"] += 1
            short_sec += p["t1"] - p["t0"]
            continue
        st[f"cut: {p['cut']}"] += 1
        name = f"chunks/{Path(g[0]['file']).stem}-{i}.wav"
        y = piece_audio(xs, offs, p["t0"], p["t1"])
        write_wav(tmp / name, y)
        src = [[c["file"], round(max(p["t0"], offs[k]) - offs[k], 3), round(min(p["t1"], offs[k + 1]) - offs[k], 3)]
               for k, c in enumerate(g) if min(p["t1"], offs[k + 1]) > max(p["t0"], offs[k])]
        first = next(c for c in g if c["file"] == src[0][0])
        last = next(c for c in g if c["file"] == src[-1][0])
        final_rows.append(dict(file=name, speaker=g[0]["speaker"], text=p["text"], duration=round(len(y) / SR, 3),
                               start=first["start"], end=last["end"],
                               **({"denoised": g[0]["denoised"]} if g[0].get("denoised") else {}), sources=src))
        made_from[name] = [c["file"] for c in g]
save_json(tmp / "metadata.json", dict(nf_meta, chunks=final_rows, sentence_chunks=dict(
    SPLIT_PAUSE_SEC=SPLIT_PAUSE_SEC, MAX_CHUNK_SEC=MAX_CHUNK_SEC, MIN_CUT_PAUSE_SEC=MIN_CUT_PAUSE_SEC,
    MIN_CHUNK_SEC=MIN_CHUNK_SEC, EDGE_SILENCE_SEC=EDGE_SILENCE_SEC, WORD_PAD=WORD_PAD)), indent=2)
if final_dir.exists():
    shutil.rmtree(final_dir)                        # the previous build
os.replace(tmp, final_dir)


def summary(cs):
    d = np.array([c["duration"] for c in cs])
    ends = sum(bool(SENT_END.search(c["text"].strip())) for c in cs)
    return (f"{len(cs):6d} chunks {d.sum() / 3600:6.2f} h | median {np.median(d):4.1f} s, 95% {np.percentile(d, 95):4.1f} s, "
            f"max {d.max():4.1f} s | end with . ! ? …: {100 * ends / len(cs):3.0f}%")


print(f"{'before (' + out_dir.name + ')':28} {summary(nf_chunks)}")
print(f"{'after (' + final_dir.name + ')':28} {summary(final_rows)}")
print(f"groups of joined chunks: {sum(len(g) > 1 for g in sgroups)} (from {sum(len(g) for g in sgroups if len(g) > 1)} chunks)")
print("pieces by the cut they start with:", dict(sorted((k, v) for k, v in st.items() if k.startswith("cut"))))
print({k: v for k, v in st.items() if not k.startswith("cut")}, f"| left out: {short_sec:.1f} s")
if sent_pauses:
    print(f"pause after a sentence end, percentiles 10/25/50/75/90: {np.round(np.percentile(sent_pauses, [10, 25, 50, 75, 90]), 2)} s"
          f" | at or over SPLIT_PAUSE_SEC: {100 * np.mean(np.array(sent_pauses) >= SPLIT_PAUSE_SEC):.0f}%")


before (dataset_nofillers)      410 chunks   1.30 h | median  6.1 s, 95% 29.5 s, max 29.5 s | end with . ! ? …:  66%
after (dataset_final)           604 chunks   1.19 h | median  3.6 s, 95% 25.4 s, max 30.0 s | end with . ! ? …:  90%
groups of joined chunks: 5 (from 97 chunks)
pieces by the cut they start with: {'cut: long': 83, 'cut: old': 9, 'cut: pause': 218, 'cut: start': 294}
{'old cuts joined': 73, 'pieces left out (short)': 24} | left out: 20.9 s
pause after a sentence end, percentiles 10/25/50/75/90: [0.   0.15 0.33 0.71 1.38] s | at or over SPLIT_PAUSE_SEC: 33%


In [11]:
# 5c. review (outputs/fillers/sentences/index.html: joined chunks before / after) and check
rev = work / "sentences"
shutil.rmtree(rev, ignore_errors=True)
joined = [g for g in sgroups if len(g) > 1]
body = []
for k, g in enumerate(joined[::max(1, len(joined) // N_REVIEW)][:N_REVIEW], 1):
    folder = rev / f"{k:02d}"
    folder.mkdir(parents=True)
    show = g[:4]                                    # a long group: its first 4 chunks and what was made of them
    files = {c["file"] for c in show}
    after = [r for r in final_rows if made_from.get(r["file"], [""])[0] == g[0]["file"] and r["sources"][0][0] in files]
    cells = []
    for side, items in (("before", [(c["file"], out_dir / c["file"], c) for c in show]),
                        ("after", [(r["file"], final_dir / r["file"], r) for r in after])):
        rows_html = []
        for i, (f, path, c) in enumerate(items):
            try:
                os.link(path, folder / f"{side}_{i}.wav")
            except OSError:
                shutil.copy2(path, folder / f"{side}_{i}.wav")
            rows_html.append(f"<p><audio controls preload='none' src='{k:02d}/{side}_{i}.wav'></audio> "
                             f"{c['duration']:.1f} s · {html.escape(c['text'])}</p>")
        cells.append(f"<td><b>{side}</b> ({len(items)})" + "".join(rows_html) + "</td>")
    more = f" (the first {len(show)} shown)" if len(g) > len(show) else ""
    body.append(f"<tr><td colspan=2><h3>{k:02d}: {html.escape(g[0]['speaker'])}, {len(g)} chunks joined{more}</h3></td></tr>"
                f"<tr>{''.join(cells)}</tr>")
(rev / "index.html").write_text(
    "<!DOCTYPE html><meta charset='utf-8'><title>Sentence chunks</title>"
    "<style>body{font-family:system-ui;margin:24px}td{vertical-align:top;padding:4px 16px 12px 0;width:50%}"
    "h3{margin:20px 0 4px}p{margin:0 0 8px;color:#333}</style><h1>Sentence chunks</h1>"
    f"<p>before = the chunks of {out_dir.name}/ that follow each other, after = the chunks of {final_dir.name}/ made from them.</p>"
    f"<table>{''.join(body)}</table>", encoding="utf-8")
print(f"{len(body)} joined groups to listen to: {rev / 'index.html'}")

# check
m3 = json.load(open(final_dir / "metadata.json", encoding="utf-8"))
wavs = {p.name for p in (final_dir / "chunks").glob("*.wav")}
names = {Path(c["file"]).name for c in m3["chunks"]}
bad = [c["file"] for c in m3["chunks"] if abs(sf.info(final_dir / c["file"]).frames / SR - c["duration"]) > 0.002]
print("every chunk has its wav:", names == wavs and len(names) == len(m3["chunks"]),
      "| durations match the wavs:", "all" if not bad else f"NO: {bad[:5]}",
      "| empty texts:", sum(not c["text"].strip() for c in m3["chunks"]),
      "| sample rate:", dict(Counter(sf.info(final_dir / c["file"]).samplerate for c in m3["chunks"][:200])))
print(f"longer than MAX_CHUNK_SEC: {sum(c['duration'] > MAX_CHUNK_SEC + 0.01 for c in m3['chunks'])} "
      f"(long speech with no sentence end to cut at, kept as it was)")
for r in [r for r in m3["chunks"] if len(r["sources"]) > 1][:3]:
    print(f"  {r['file']}  {r['duration']:.1f} s from {len(r['sources'])} chunks | {r['text'][:90]}")


5 joined groups to listen to: /home/intern/projects/tts/runs/test/outputs/fillers/sentences/index.html
every chunk has its wav: True | durations match the wavs: all | empty texts: 0 | sample rate: {48000: 200}
longer than MAX_CHUNK_SEC: 0 (long speech with no sentence end to cut at, kept as it was)
  chunks/001_0000_speaker_0-1.wav  6.0 s from 2 chunks | Neyrobiologiyaga, o'zi miyaga qiziqishim azaldan bo'lgan, bu narsa haqida gapirganman.
  chunks/001_0000_speaker_0-3.wav  23.2 s from 2 chunks | I hozirgi kunda neyrobiologiyaga qiziqayotganlar o'sha podkastlarimni ko'rib, ko'chalarda 
  chunks/001_0000_speaker_0-4.wav  20.8 s from 2 chunks | Ilm yo'lidan ketyapti. O'zini yaxshiroq anglashga harakat qilyapti va nima deydi, vaqtini 
